<a href="https://colab.research.google.com/github/RSC-SC/ML-ProjetoFinal/blob/fase%2Ffeature-engineering/ML_Projeeto_Final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Projeto Final

## Carregando dados

In [ ]:
import pandas as pd
# Para carregar os dados precisa fazer upload do arquivo credit_risk_dataset.csv para /home
# Load the CSV file into a DataFrame
df = pd.read_csv('/home/credit_risk_dataset.csv')

# Display the first few rows of the DataFrame
print(df.head())

## Traduzir colunas

In [ ]:
# Define a dictionary for column name translation
translation_map = {
    'person_age': 'idade_pessoa',
    'person_income': 'renda_pessoa',
    'person_home_ownership': 'propriedade_casa_pessoa',
    'person_emp_length': 'tempo_emprego_pessoa',
    'loan_intent': 'intencao_emprestimo',
    'loan_grade': 'grau_emprestimo',
    'loan_amnt': 'valor_emprestimo',
    'loan_int_rate': 'taxa_juros_emprestimo',
    'loan_status': 'status_emprestimo',
    'loan_percent_income': 'porcentagem_renda_emprestimo',
    'cb_person_default_on_file': 'inadimplencia_historico_credito',
    'cb_person_cred_hist_length': 'tempo_historico_credito_pessoa'
}

# Rename the columns
df = df.rename(columns=translation_map)

# Display the new column names to verify
print("Novas colunas do DataFrame:", df.columns.tolist())
print(df.head())

## Fase 1: Análise Exploratória de Dados (EDA)

### Descritiva e Estatística

In [ ]:
# 1. Exibir o tamanho total da base (linhas e colunas)
print("Tamanho total da base (linhas, colunas):")
print(df.shape)

print("\n")

# 2. Exibir os tipos de dados de cada variável
print("Tipos de dados de cada variável:")
print(df.dtypes)

print("\n")

# 3. Exibir o sumário estatístico descritivo para colunas numéricas
print("Sumário estatístico descritivo para variáveis numéricas:")
print(df.describe())

print("\n")

# 4. Exibir o sumário estatístico descritivo para colunas categóricas (opcional, mas útil)
print("Sumário estatístico descritivo para variáveis categóricas:")
print(df.describe(include='object'))

### Visual

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Definir o estilo dos gráficos
sns.set_style("whitegrid")

# 1. Histograma de Distribuição de Idades
plt.figure(figsize=(10, 6))
sns.histplot(df['idade_pessoa'], bins=30, kde=True)
plt.title('Distribuição da Idade da Pessoa')
plt.xlabel('Idade')
plt.ylabel('Frequência')
plt.show()

# Descrição do Histograma de Idades
print("Este histograma mostra a distribuição da idade dos indivíduos no dataset.")
print("Podemos observar a concentração de idades e identificar possíveis outliers")
print("ou grupos etários proeminentes.\n")

# 2. Gráfico de Barras do Desbalanceamento da Variável Alvo (status_emprestimo)
plt.figure(figsize=(8, 5))
sns.countplot(x='status_emprestimo', hue='status_emprestimo', data=df, palette='viridis', legend=False)
plt.title('Desbalanceamento da Variável Alvo: Status do Empréstimo')
plt.xlabel('Status do Empréstimo (0: Pago, 1: Inadimplente)')
plt.ylabel('Contagem')
plt.xticks(ticks=[0, 1], labels=['Pago', 'Inadimplente'])
plt.show()

# Descrição do Gráfico de Barras
print("Este gráfico de barras ilustra a contagem de empréstimos pagos (0) e")
print("inadimplentes (1). É possível verificar o desbalanceamento da variável alvo,")
print("onde há significativamente mais empréstimos pagos do que inadimplentes, o que é")
print("comum em problemas de risco de crédito e deve ser considerado na modelagem.\n")

# 3. Mapa de Calor da Correlação de Pearson
plt.figure(figsize=(12, 10))
# Calcular a matriz de correlação apenas para colunas numéricas
correlation_matrix = df.select_dtypes(include=['number']).corr()
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt=".2f", linewidths=.5)
plt.title('Mapa de Calor da Correlação de Pearson')
plt.show()

# Descrição do Mapa de Calor
print("O mapa de calor acima exibe a matriz de correlação de Pearson entre as variáveis")
print("numéricas do dataset. Ele ajuda a identificar relações lineares entre pares de variáveis.")
print("Cores mais quentes (vermelhas) indicam correlação positiva forte, enquanto cores mais")
print("frias (azuis) indicam correlação negativa forte. Valores próximos de zero indicam pouca")
print("ou nenhuma correlação linear.\n")

### Tomada de Decisão

A análise revelou que a base de dados possui um desbalanceamento significativo na variável alvo (status_emprestimo), com uma predominância de empréstimos pagos, o que exigirá técnicas de reamostragem (como SMOTE ou undersampling) para evitar modelos tendenciosos. O histograma de idade mostrou uma concentração em jovens adultos, mas com a presença de outliers (idades extremas como 144 anos) que precisam ser tratados ou removidos. A matriz de correlação indicou que, embora existam relações lineares, nenhuma variável isolada explica totalmente a inadimplência, sugerindo que a combinação de fatores através de modelos não-lineares será necessária. Além disso, a presença de valores ausentes identificada no describe() (em taxa_juros_emprestimo e tempo_emprego_pessoa) guiará a estratégia de imputação, garantindo que a qualidade dos dados seja mantida para a fase de modelagem.

## Fase 2: Tratamento e Limpeza (Data Prep)

### Duplicados

In [ ]:
print(f"Número de linhas antes de remover duplicatas: {df.shape[0]}")

# Remover linhas duplicadas
df.drop_duplicates(inplace=True)

print(f"Número de linhas após remover duplicatas: {df.shape[0]}")

### Valores Nulos:

In [ ]:
print("Colunas que *tinham* valores ausentes e a técnica de imputação:")
print("----------------------------------------------------------------")
print("tempo_emprego_pessoa: Mediana")
print("taxa_juros_emprestimo: Mediana")
print("\nJustificativa:\n")
print("A mediana foi escolhida para a imputação de valores ausentes nas colunas 'tempo_emprego_pessoa' e 'taxa_juros_emprestimo'.")
print("Esta técnica é preferível em situações onde as distribuições dos dados podem ser assimétricas ou conter outliers, como foi observado previamente em 'idade_pessoa' e 'tempo_emprego_pessoa'.")
print("A mediana é menos sensível a valores extremos do que a média, o que ajuda a evitar que a imputação introduza um viés indevido ou distorça as estatísticas da coluna.")
print("Isso garante que os dados preenchidos sejam mais representativos da tendência central dos valores existentes, mitigando o impacto de possíveis anomalias nos dados originais.")

# Confirmar que não há mais valores nulos (opcional, pois já foi verificado antes)
print("\nVerificação final de valores nulos após imputação:")
print(df.isnull().sum()[df.isnull().sum() > 0])

### Tratamento de Outliers:

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

# Identificar colunas numéricas para tratamento de outliers
numerical_cols_outliers = [
    'idade_pessoa',
    'renda_pessoa',
    'tempo_emprego_pessoa',
    'valor_emprestimo',
    'taxa_juros_emprestimo',
    'porcentagem_renda_emprestimo',
    'tempo_historico_credito_pessoa'
]

print("### Identificação de Outliers via Boxplots ###\n")

# Gerar boxplots para visualizar outliers
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols_outliers):
    plt.subplot(3, 3, i + 1)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot de {col}')
    plt.ylabel('') # Remova o rótulo y padrão, pois o título já é descritivo o suficiente
plt.tight_layout()
plt.show()

print("\n### Análise e Proposta de Tratamento de Outliers ###\n")
print("Após a análise dos boxplots, é evidente a presença de outliers significativos em várias colunas numéricas, especialmente em 'idade_pessoa', 'renda_pessoa', 'tempo_emprego_pessoa', 'valor_emprestimo' e 'porcentagem_renda_emprestimo'.")
print("Os valores extremos, como idades de 144 anos ou tempos de emprego de 123 anos, são provavelmente erros de entrada ou dados atípicos que podem distorcer a análise e o treinamento de modelos.")

print("\n#### Estratégia de Tratamento: Clipping (Limitação) ####")
print("Dado que modelos como o K-Nearest Neighbors (KNN) são extremamente sensíveis a outliers, pois eles podem afetar drasticamente o cálculo de distâncias euclidianas, e para manter a maior parte dos dados válidos, optarei pela técnica de *clipping* (limitação). Essa técnica consiste em "
      "substituir valores que estão acima de um limite superior ou abaixo de um limite inferior por esses próprios limites, ao invés de removê-los completamente. Isso ajuda a reduzir o impacto dos outliers sem a perda de dados. Para cada coluna, utilizarei o método do IQR (Intervalo Interquartil) para definir os limites.")

print("\n#### Implementação do Clipping usando o método IQR ####")

# Função para aplicar clipping com base no IQR
def clip_outliers_iqr(df, column):
    Q1 = df[column].quantile(0.25)
    Q3 = df[column].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    # Alguns outliers superiores parecem ser legítimos (ex: renda alta), então ajustaremos alguns limites
    # Idade: Limitar para um máximo mais razoável, e um mínimo de 18 anos
    if column == 'idade_pessoa':
        lower_bound = max(lower_bound, 18) # Idade mínima razoável
        upper_bound = min(upper_bound, 80) # Idade máxima razoável
    # Tempo de Emprego: Limitar para um máximo razoável
    elif column == 'tempo_emprego_pessoa':
        upper_bound = min(upper_bound, 50) # Tempo de emprego máximo razoável

    # Aplicar o clipping
    df[column] = np.where(df[column] < lower_bound, lower_bound, df[column])
    df[column] = np.where(df[column] > upper_bound, upper_bound, df[column])
    return df

# Aplicar clipping nas colunas identificadas
for col in numerical_cols_outliers:
    df = clip_outliers_iqr(df.copy(), col)
    print(f"Clipping aplicado na coluna: {col}")

print("\n### Verificação dos Dados Após Clipping ###")
print(df[numerical_cols_outliers].describe())

# Gerar boxplots novamente para verificar o efeito do clipping
print("\n### Boxplots após o Clipping ###\n")
plt.figure(figsize=(15, 10))
for i, col in enumerate(numerical_cols_outliers):
    plt.subplot(3, 3, i + 1)
    sns.boxplot(y=df[col])
    plt.title(f'Boxplot de {col} (Após Clipping)')
    plt.ylabel('')
plt.tight_layout()
plt.show()

print("O clipping foi aplicado para mitigar o impacto dos outliers, tornando os dados mais adequados para modelos sensíveis como o KNN, sem descartar as observações e perdendo informações.")
print("Para modelos mais robustos a outliers, como Árvores de Decisão, o impacto do clipping pode ser menor, mas ainda contribui para uma representação mais limpa e plausível dos dados.")

## Fase 3: Feature Engineering (Coluna Calculada)

In [ ]:
# Verificar se há 'renda_pessoa' igual a zero para evitar divisão por zero
# Pelas estatísticas descritivas anteriores, o valor mínimo de 'renda_pessoa' é 4000, então não haverá divisão por zero.
if (df['renda_pessoa'] == 0).any():
    print("Aviso: 'renda_pessoa' contém valores zero. Realizando tratamento para evitar divisão por zero.")
    # Para evitar divisões por zero, podemos substituir 0 por um valor muito pequeno ou NaN para tratamento posterior.
    df['renda_pessoa'] = df['renda_pessoa'].replace(0, np.nan) # Substitui 0 por NaN
    df['renda_pessoa'] = df['renda_pessoa'].fillna(df['renda_pessoa'].median()) # Coloca NaN com a mediana

# Criar a nova coluna 'comprometimento_renda'
df['comprometimento_renda'] = (df['valor_emprestimo'] / df['renda_pessoa']) * 100

print("Nova coluna 'comprometimento_renda' criada com sucesso.")
print("Verificando as primeiras linhas com a nova coluna:")
print(df[['valor_emprestimo', 'renda_pessoa', 'comprometimento_renda']].head())

print("\nEstatísticas descritivas da nova coluna:")
print(df['comprometimento_renda'].describe())